<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **7. Wi-Fi Simulation with Jemula802**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 7.1 Introduction

Jemula802 is a Java simulator for studying IEEE 802.11 medium access control (MAC). It represents packet transmissions, channel access, traffic generation, and radio conditions through configurable models. Scenario settings are stored in XML files.

This notebook runs a small example and compares throughput and delay for two access categories. It introduces the workflow used in the other Wi-Fi simulation notebooks: prepare the simulator, select a scenario, run it, and read the result files.

The following cell downloads or updates Jemula802 and its supporting Jemula library. Compilation requires a Java Development Kit (JDK).


In [ ]:
classpath_jemula = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar"
classpath_jemula802 = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar:jemula/lib/commons-io-2.14.0.jar"

import platform

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")


In [ ]:
%%bash
set -e
if [ -d "jemula802/.git" ]; then
  echo "Updating existing repo jemula802 ..."
  git -C jemula802 pull --ff-only
else
  echo "Cloning repo jemula802 ..."
  git clone https://bitbucket.org/lfield/jemula802.git
fi
set -e
if [ -d "jemula/.git" ]; then
  echo "Updating existing repo jemula ..."
  git -C jemula pull --ff-only
else
  echo "Cloning repo jemula ..."
  git clone https://bitbucket.org/lfield/jemula.git
fi


### 7.2 Compile the Java code


In [ ]:
!javac -g:none -classpath $classpath_jemula -d "jemula/bin" jemula/src/plot/*.java jemula/src/kernel/*.java jemula/src/statistics/*.java
!echo Compilation of jemula done.

!javac -g:none  -classpath $classpath_jemula802 -d "jemula802/bin" jemula802/src/emulator/*.java jemula802/src/layer0_medium/*.java jemula802/src/layer1_802Phy/*.java \
 jemula802/src/layer1_80211Phy/*.java jemula802/src/layer2_802Mac/*.java jemula802/src/layer2_80211Mac/*.java \
 jemula802/src/layer2_802Algorithms/*.java jemula802/src/layer3_network/*.java jemula802/src/layer5_application/*.java \
 jemula802/src/layer4_transport/*.java jemula802/src/station/*.java jemula802/src/visualization/*.java \
 jemula802/src/util/*.java jemula802/src/gui/*.java
!echo Compilation of jemula802 done.

import os

ROOT_DIR = os.getcwd()
!echo Root directory is $ROOT_DIR

### 7.3 Run the example

[Jemula802GettingStarted.xml](jemula802/scenarios/examples/Jemula802GettingStarted.xml) defines two radio stations and traffic in two access categories, AC1 and AC2. The categories have separate queues and channel-access settings. Their numbers are simulator identifiers; the XML configuration determines their priority and offered load.


In [ ]:
EXAMPLE_SCENARIO="jemula802/scenarios/examples/Jemula802GettingStarted.xml"
!echo Running example scenario $EXAMPLE_SCENARIO
!java -Xmx500m -classpath $classpath_jemula802 emulator.JE802Starter $EXAMPLE_SCENARIO

### 7.4 Inspect the results

#### 7.4.1 Result files

The simulation creates `results_GettingStarted`, containing statistics and a copy of the XML scenario. The files distinguish offered load, delivered throughput, and packet delay.

#### 7.4.2 Optional map visualization

The generated KML file describes the scenario's positions and can be opened in a compatible map viewer, such as Google Earth Pro. This visualization is optional; the statistical analysis below uses the CSV files.

#### 7.4.3 Statistical analysis

The first plot combines offered load and throughput in Mb/s with average delay in ms. The second shows the delay distribution. The descending cumulative curve represents the fraction of recorded delays above a threshold, or complementary cumulative distribution function (CCDF); its current legend says CDF.

The following cells plot all traffic first, then AC1 and AC2 separately.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

if os.path.isdir("results_GettingStarted"):
    os.chdir("results_GettingStarted")

In [ ]:
def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6], '-', color='g', linewidth=4, label='Offer [Mb/s]')
    plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6], '-', color='w', linewidth=1, label='Throughput [Mb/s]')
    plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3], '-', color='r', linewidth=1, label='Avg. Delay [ms]')
    #plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 5], '-', color='r', linewidth=1)
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(aBinSize_ms_string, aHistogram_string):
    fig, ax_distr = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))
    theHistogram = [float(x) for x in aHistogram_string.split()]
    theHistogram = [x/sum(theHistogram) for x in theHistogram]
    theNumOfBins = len(theHistogram)
    theBinSize_ms = float(aBinSize_ms_string)
    theX = [i * theBinSize_ms + theBinSize_ms/2 for i in range(theNumOfBins)] 
    theCumSum = 1-np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)
    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)  # Set x-axis limits
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay>=x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

Results for all traffic:


In [ ]:
if os.path.isdir("results_GettingStarted"):
    os.chdir("results_GettingStarted")
result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)
plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])


Enhanced Distributed Channel Access (EDCA) provides differentiated access through separate queues and backoff settings. Its parameters include the Arbitration Inter-Frame Space Number (AIFSN), minimum and maximum contention-window bounds (CWmin and CWmax), and transmission opportunity (TXOP) limit.

Smaller waiting intervals and contention windows can give a category more frequent access, but do not guarantee a particular throughput or delay. Offered load also matters. Compare AC1 and AC2 with their XML settings in mind.

Results for AC1:


In [ ]:
if os.path.isdir("results_GettingStarted"):
    os.chdir("results_GettingStarted")
result_AC1_offer = pd.read_csv('offer_AC1.csv', skiprows=2)
result_AC1_thrp = pd.read_csv('thrp_AC1.csv', skiprows=2)
result_AC1_delay = pd.read_csv('delay_AC1.csv', skiprows=2)
plot_thrp_offer_delay(result_AC1_offer, result_AC1_thrp, result_AC1_delay)
plot_delay_histogram(result_AC1_delay.iloc[-1,7],result_AC1_delay.iloc[-1,8])

Results for AC2:


In [ ]:

if os.path.isdir("results_Jemula802GettingStarted"):
    os.chdir("results_Jemula802GettingStarted")
result_AC2_offer = pd.read_csv('offer_AC2.csv', skiprows=2)
result_AC2_thrp = pd.read_csv('thrp_AC2.csv', skiprows=2)
result_AC2_delay = pd.read_csv('delay_AC2.csv', skiprows=2)
plot_thrp_offer_delay(result_AC2_offer, result_AC2_thrp, result_AC2_delay)
plot_delay_histogram(result_AC2_delay.iloc[-1,7],result_AC2_delay.iloc[-1,8])

Total throughput includes traffic from both categories. Total average delay is an average over the packets included in that statistic, so it need not equal the simple mean of the two category averages. Differences in offered load and delivered packet counts affect the comparison.


----
### 7.5 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**How does total throughput relate to the throughputs of AC1 and AC2?**

Answer:


**How do the average delays of AC1 and AC2 compare? Why need total average delay not equal their simple arithmetic mean?**

Answer:


**Which access category has more favorable channel-access settings in the XML scenario? Do the throughput and delay results reflect that advantage?**

Answer:


**What does the `seed` parameter in [Jemula802GettingStarted.xml](jemula802/scenarios/examples/Jemula802GettingStarted.xml) control? Which results might change if the seed changes while the other settings remain fixed?**

Answer:


**How do radio regulation and technical standardization differ? Why can manufacturers use proprietary algorithms while still complying with a wireless standard?**

Answer:


**OPTIONAL: Are all explanations and exercises in this notebook clear? Which parts, if any, need clarification or improvement?**

Answer:
